# Faraday MR — corrida exhaustiva en GPU (Colab)

Proyecto III, foreground galáctico: disco de NE2001 + 4 brazos espirales,
campo regular de disco (espiral con inversión interior) y halo de JF12,
turbulencia de Kolmogorov, ray tracing desde la posición del Sol, comparación
contra Oppermann+2012, NVSS (Taylor+2009), Haslam 408 MHz y Planck 30 GHz, y
remoción del foreground polarizado de Planck por ajuste de plantilla.

Corre el **perfil "exhaustivo"** de `config_fisica.py`:

| | rápido (CPU) | exhaustivo (este notebook) |
|---|---|---|
| Malla 3D | 64³, celdas de 0.5 kpc | 256³, celdas de 0.125 kpc |
| Escala mínima de la turbulencia | 1 kpc | 0.25 kpc |
| Mapa de cielo | 180 × 91 (2°) | 360 × 181 (1°) |
| Paso de integración | 0.1 kpc | 0.025 kpc |

**Runtime:** `Entorno de ejecución → Cambiar tipo de entorno → GPU` (T4 alcanza;
L4/A100 van más rápido). Ejecutar las celdas en orden. Si alguna celda
importa `config` antes de la celda 6, hay que reiniciar el runtime: el perfil
se lee una sola vez, al importar `config_fisica.py`.

## 1. GPU asignada

In [ ]:
!nvidia-smi

## 2. Traer el código

Colab no ve tu disco: el código tiene que estar en GitHub o subirse como zip.

- **`ORIGEN = "github"`**: clona `BRANCH`. Solo sirve si ya hiciste push de
  los últimos cambios.
- **`ORIGEN = "zip"`**: sube un zip de la carpeta del repo (con el panel de
  archivos de la izquierda, a `/content/`) y pon su nombre en `ZIP_SUBIDO`.

In [ ]:
import os
import zipfile

ORIGEN = "github"  # "github" o "zip"
REPO_URL = "https://github.com/hacostah/Mapas-de-Rotacion-de-Faraday"
BRANCH = "V4-Foreground-CMB"
ZIP_SUBIDO = "/content/Mapas-de-Rotacion-de-Faraday.zip"
RUTA_REPO = "/content/Mapas-de-Rotacion-de-Faraday"

if os.path.isdir(RUTA_REPO):
    print(f"Ya existe {RUTA_REPO}; se usa tal cual (bórralo para traer una copia nueva).")
elif ORIGEN == "github":
    !git clone --branch {BRANCH} {REPO_URL} {RUTA_REPO}
elif ORIGEN == "zip":
    with zipfile.ZipFile(ZIP_SUBIDO) as zf:
        zf.extractall("/content/")
    if not os.path.isdir(RUTA_REPO):
        raise FileNotFoundError(
            f"El zip no creó {RUTA_REPO}: tiene que contener la carpeta "
            "Mapas-de-Rotacion-de-Faraday en su raíz."
        )
else:
    raise ValueError("ORIGEN debe ser 'github' o 'zip'.")

%cd {RUTA_REPO}

# El código de esta versión tiene la remoción contra Planck; si no está, se
# clonó una versión vieja (falta hacer push).
with open("faradaymr/observational.py") as f:
    assert "def remover_foreground_polarizado_planck" in f.read(), (
        "Esta copia del repo es anterior a la versión del póster: haz push de "
        "los cambios (o usa ORIGEN='zip')."
    )
print("Código OK.")

## 3. Dependencias

`cupy` se instala con la build del CUDA de la imagen de Colab.

In [ ]:
import subprocess

!pip install -q -r requirements.txt

salida_nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout
paquete_cupy = "cupy-cuda11x" if "release 11" in salida_nvcc else "cupy-cuda12x"
print(f"Instalando {paquete_cupy} ...")
!pip install -q {paquete_cupy}

## 4. Comprobar el backend GPU

Integra un cielo pequeño con numpy y con cupy, y exige que den lo mismo:
así un error de la ruta GPU aparece aquí y no a mitad de la corrida pesada.

In [ ]:
import numpy as np

from faradaymr import get_backend, los_raytrace, to_numpy

cp = get_backend(use_gpu=True)  # falla aquí si cupy/CUDA no quedaron bien
print("Backend:", cp.__name__, "| GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())

rng = np.random.default_rng(0)
n, dx = 24, 1.0
campos = [rng.normal(size=(n, n, n)) for _ in range(3)] + [np.abs(rng.normal(size=(n, n, n))) + 0.1] * 2
l = np.linspace(-np.pi, np.pi, 12, endpoint=False)
b = np.linspace(-1.2, 1.2, 7)
args = dict(dl=0.25, frequency=1.4e9, wavelength=0.214, p_index=3.0, length_unit_pc=1000.0)
obs_pos = np.array([8.0, 12.0, 12.0])

en_cpu = los_raytrace.sky_map(*campos, obs_pos, dx, n * dx, l, b, xp=np, **args)
en_gpu = los_raytrace.sky_map(*[cp.asarray(c) for c in campos], cp.asarray(obs_pos), dx, n * dx,
                              cp.asarray(l), cp.asarray(b), xp=cp, **args)
for nombre, a, g in zip(("RM", "I", "Q", "U"), en_cpu, en_gpu):
    np.testing.assert_allclose(to_numpy(g), a, rtol=1e-6, atol=1e-12 * np.abs(a).max())
    print(f"{nombre}: CPU y GPU coinciden")

## 5. Datos observacionales

~530 MB (Oppermann+2012, NVSS Taylor+2009, Haslam 408 MHz, Planck 30 GHz) en
`data/external/`.

In [ ]:
!python data/external/descargar_datos.py

## 6. Perfil exhaustivo y tamaño de lote

Fija el perfil y elige cuántos píxeles del cielo se integran a la vez según
la memoria libre de la GPU (~220 KB por píxel en este perfil, más ~6 GB para
los campos 256³ y sus copias). **Tiene que correr antes de importar
`config`.**

In [ ]:
import os

import cupy as cp

BYTES_POR_PIXEL = 220e3
RESERVA_CAMPOS_BYTES = 6e9

libre, total = cp.cuda.runtime.memGetInfo()
lote = int(0.6 * (libre - RESERVA_CAMPOS_BYTES) / BYTES_POR_PIXEL)
lote = max(1024, min(32768, lote // 1024 * 1024))
print(f"GPU: {libre / 1e9:.1f} GB libres de {total / 1e9:.1f} GB -> lote de {lote} píxeles")

os.environ["FARADAYMR_PERFIL_RESOLUCION"] = "exhaustivo"
os.environ["FARADAYMR_PIXEL_CHUNK_SIZE"] = str(lote)

import config

print("Perfil:", config.cfg_units.PERFIL_RESOLUCION)
print(f"Malla {config.N_BASE}³ (dx = {config.DX_BASE_KPC} kpc), cielo {config.N_L} x {config.N_B}, "
      f"paso {config.DL_KPC} kpc, lote {config.PIXEL_CHUNK_SIZE}")
print(f"Turbulencia efectiva: {config.B0_TURBULENTO_MG:.3f} µG "
      f"(= 3 µG x {config.FACTOR_TURBULENCIA:.3f}, derivada de la malla)")
assert config.N_BASE == 256, "No quedó el perfil exhaustivo: reinicia el runtime y corre desde la celda 1."

## 7. Recalibrar el campo regular a esta resolución

`FACTOR_CAMPO_REGULAR = 1.0` se calibró en el perfil rápido. Con brazos,
disco y turbulencia mejor resueltos el óptimo puede moverse; este paso repite
**el mismo procedimiento** (`calibrar_amplitud_campo.py`: ajuste contra
Oppermann+2012 y NVSS, media geométrica) a esta resolución. Con
`APLICAR_RECALIBRACION = True` el valor nuevo se usa en la corrida (solo en
memoria; `config_fisica.py` no se toca, y el valor queda registrado en
`parametros_corrida.json`).

In [ ]:
import time

import cupy as cp

from calibrar_amplitud_campo import calibrar

APLICAR_RECALIBRACION = True

t0 = time.perf_counter()
calibracion = calibrar(seed=0, use_gpu=True)
tiempo_calibracion = time.perf_counter() - t0
cp.get_default_memory_pool().free_all_blocks()

factor_rapido = config.FACTOR_CAMPO_REGULAR
if APLICAR_RECALIBRACION:
    config.FACTOR_CAMPO_REGULAR = round(calibracion["factor_combinado"], 2)
print(f"\nFactor del perfil rápido: {factor_rapido:.2f} | usado en esta corrida: "
      f"{config.FACTOR_CAMPO_REGULAR:.2f} | calibración en {tiempo_calibracion:.0f} s")

## 8. Simulación en GPU

Escenario 256³, mapas de cielo a 1.4 GHz y 28.4 GHz, y 8 realizaciones de la
turbulencia (varianza galáctica para Planck y signo de la RM). `use_gpu=True`
explícito: si algo falla con cupy, falla aquí en vez de caer en silencio a CPU.

In [ ]:
import time

from run import ejecutar_corrida

t0 = time.perf_counter()
ejecutar_corrida(use_gpu=True, seed=0)
tiempo_simulacion = time.perf_counter() - t0
print(f"Simulación completa en {tiempo_simulacion / 60:.1f} min")

## 9. Comparación con datos reales y remoción del foreground de Planck

In [ ]:
import time

from compare_observaciones import ejecutar_comparacion

t0 = time.perf_counter()
resumen = ejecutar_comparacion()
tiempo_comparacion = time.perf_counter() - t0

## 10. Resumen

In [ ]:
import json

import cupy as cp

opp = resumen["oppermann_2012"]
nvss = resumen["catalogo_taylor_2009"]
planck = resumen["validacion_planck_030ghz"]
remocion = resumen["remocion_foreground_planck_030ghz"]
signos = resumen["signo_rm_por_region"]

print(f"RMS(RM) global: modelo {opp['rms_modelo_global_rad_m2']:.0f} | Oppermann {opp['rms_observado_global_rad_m2']:.0f} rad/m²")
print("RMS(RM) por |b| (modelo / Oppermann / NVSS sin ruido):")
for banda in opp["perfil_rms_vs_b"]:
    nv = banda["rms_nvss_sin_ruido"]
    print(f"  |b|={banda['b_deg']:4.1f}°: {banda['rms_modelo']:6.1f} / {banda['rms_oppermann']:6.1f} / "
          f"{'—' if nv is None else f'{nv:6.1f}'}")
print(f"NVSS: r = {nvss['correlacion_pearson']:.2f} en {nvss['n_fuentes_comparadas']} fuentes")
print(f"Signos de RM correctos: {sum(v['signo_correcto'] for v in signos.values())}/{len(signos)}")
print(f"Haslam: r(log-log) = {resumen['morfologia_haslam_408mhz']['correlacion_log_pearson']:.2f}")
ang, forma = planck["angulo_polarizacion"], planck["forma_intensidad_polarizada"]
print(f"Planck: <cos 2Δψ> = {ang['coherencia_modelo_media']:.2f} ± {ang['coherencia_modelo_dispersion_entre_realizaciones']:.2f} "
      f"({'pasa' if ang['pasa'] else 'no pasa'}); R²(P) = {forma['r2_modelo_media']:.2f} "
      f"vs disco {forma['r2_disco_plano_paralelo']:.2f} ({'pasa' if forma['pasa'] else 'no pasa'})")
print("Remoción del foreground polarizado de Planck (modelo | trivial):")
for nombre, region in remocion["regiones"].items():
    print(f"  {nombre:16s} {100 * region['fraccion_removida_modelo']:5.1f} % | "
          f"{100 * region['fraccion_removida_plantilla_trivial']:5.1f} %  "
          f"({100 * region['fraccion_potencia_planck']:.0f} % de la potencia)")

parametros = {
    "perfil": config.cfg_units.PERFIL_RESOLUCION,
    "n_base": config.N_BASE,
    "dx_kpc": config.DX_BASE_KPC,
    "cielo": [config.N_L, config.N_B],
    "paso_kpc": config.DL_KPC,
    "pixel_chunk_size": config.PIXEL_CHUNK_SIZE,
    "factor_campo_regular_perfil_rapido": factor_rapido,
    "factor_campo_regular_usado": config.FACTOR_CAMPO_REGULAR,
    "calibracion_a_esta_resolucion": calibracion,
    "b0_turbulento_efectivo_uG": config.B0_TURBULENTO_MG,
    "gpu": cp.cuda.runtime.getDeviceProperties(0)["name"].decode(),
    "tiempo_s": {"calibracion": tiempo_calibracion, "simulacion": tiempo_simulacion,
                 "comparacion": tiempo_comparacion},
}
with open("results/foreground_galactico/parametros_corrida.json", "w") as f:
    json.dump(parametros, f, indent=2, ensure_ascii=False)
print("\nParámetros guardados en results/foreground_galactico/parametros_corrida.json")

## 11. Figuras principales

In [ ]:
from IPython.display import Image, display

for nombre in [
    "mapa_de_cielo.png",
    "comparacion_perfil_latitud.png",
    "comparacion_rm_vs_longitud.png",
    "remocion_foreground_planck.png",
    "validacion_planck_030ghz.png",
    "estadistica_mapa.png",
]:
    print(nombre)
    display(Image(filename=f"results/foreground_galactico/{nombre}", width=1100))

## 12. Descargar los resultados

In [ ]:
import shutil

ruta_zip = shutil.make_archive("/content/resultados_exhaustivo_gpu", "zip", "results")
print(f"Resultados en {ruta_zip}")
try:
    from google.colab import files

    files.download(ruta_zip)
except ImportError:
    print("Fuera de Colab: descarga el zip de la ruta de arriba.")